In [1]:
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from utils import compute_pass_metrics, expand_tool_usage, load_all_results

# Configuration
BASELINE_NAME = "haiku-4-5"
ALTOOL_NAME = "haiku-4-5-altool"
K_RUNS = 10  # Use first 10 runs for fair comparison

# Load data
all_results = load_all_results()
baseline_df = all_results[BASELINE_NAME]
altool_df = all_results[ALTOOL_NAME]

print(f"Loaded {BASELINE_NAME}: {baseline_df['run_id'].nunique()} runs, {baseline_df['instance_id'].nunique()} instances")
print(f"Loaded {ALTOOL_NAME}: {altool_df['run_id'].nunique()} runs, {altool_df['instance_id'].nunique()} instances")

Loaded haiku-4-5: 11 runs, 55 instances
Loaded haiku-4-5-altool: 10 runs, 55 instances


In [2]:
# Filter to first K_RUNS for fair comparison
def filter_to_k_runs(df: pd.DataFrame, k: int) -> pd.DataFrame:
    run_ids = sorted(df["run_id"].unique())[:k]
    return df[df["run_id"].isin(run_ids)].copy()


baseline_df_k = filter_to_k_runs(baseline_df, K_RUNS)
altool_df_k = filter_to_k_runs(altool_df, K_RUNS)

print(f"Using first {K_RUNS} runs for comparison:")
print(f"  {BASELINE_NAME}: {baseline_df_k['run_id'].nunique()} runs, {len(baseline_df_k)} total records")
print(f"  {ALTOOL_NAME}: {altool_df_k['run_id'].nunique()} runs, {len(altool_df_k)} total records")

Using first 10 runs for comparison:
  haiku-4-5: 10 runs, 550 total records
  haiku-4-5-altool: 10 runs, 550 total records


## 1. Resolution Rate Comparison

We compare mean resolution rates using bootstrap confidence intervals and a permutation test for statistical significance.

In [ ]:
def bootstrap_ci(data: np.ndarray, n_bootstrap: int = 10000, ci_level: float = 0.95) -> dict[str, Any]:
    """Compute bootstrap confidence interval using percentile method."""
    rng = np.random.default_rng(42)
    bootstrap_means = np.array([rng.choice(data, size=len(data), replace=True).mean() for _ in range(n_bootstrap)])
    alpha = 1 - ci_level
    ci_low = np.percentile(bootstrap_means, alpha / 2 * 100)
    ci_high = np.percentile(bootstrap_means, (1 - alpha / 2) * 100)
    return {
        "mean": float(data.mean()),
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
        "ci_half": float((ci_high - ci_low) / 2),
        "bootstrap_means": bootstrap_means,
    }


def permutation_test(data1: np.ndarray, data2: np.ndarray, n_permutations: int = 10000) -> dict[str, float]:
    """Two-sample permutation test for difference in means."""
    rng = np.random.default_rng(42)
    observed_diff = data1.mean() - data2.mean()
    combined = np.concatenate([data1, data2])
    n1 = len(data1)

    perm_diffs = []
    for _ in range(n_permutations):
        rng.shuffle(combined)
        perm_diff = combined[:n1].mean() - combined[n1:].mean()
        perm_diffs.append(perm_diff)

    perm_diffs = np.array(perm_diffs)
    # Two-tailed p-value
    p_value = (np.abs(perm_diffs) >= np.abs(observed_diff)).mean()

    return {
        "observed_diff": float(observed_diff),
        "p_value": float(p_value),
        "perm_diffs": perm_diffs,  # type: ignore
    }


def cohens_d(data1: np.ndarray, data2: np.ndarray) -> float:
    """Compute Cohen's d effect size."""
    n1, n2 = len(data1), len(data2)
    var1, var2 = data1.var(ddof=1), data2.var(ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return float((data1.mean() - data2.mean()) / pooled_std) if pooled_std > 0 else 0.0

In [4]:
# Compute per-run resolution rates
baseline_per_run = baseline_df_k.groupby("run_id")["resolved"].mean().values * 100  # type: ignore
altool_per_run = altool_df_k.groupby("run_id")["resolved"].mean().values * 100  # type: ignore

# Bootstrap CIs
baseline_boot = bootstrap_ci(np.array(baseline_per_run))
altool_boot = bootstrap_ci(np.array(altool_per_run))

# Permutation test
perm_result = permutation_test(np.array(altool_per_run), np.array(baseline_per_run))

# Effect size
effect_size = cohens_d(np.array(altool_per_run), np.array(baseline_per_run))

print("Resolution Rate Comparison (Mean ± 95% CI)")
print("=" * 60)
print(f"{BASELINE_NAME:<20} {baseline_boot['mean']:>6.2f}% ({baseline_boot['ci_low']:.2f}% - {baseline_boot['ci_high']:.2f}%)")
print(f"{ALTOOL_NAME:<20} {altool_boot['mean']:>6.2f}% ({altool_boot['ci_low']:.2f}% - {altool_boot['ci_high']:.2f}%)")
print("-" * 60)
print(f"Difference (altool - baseline): {perm_result['observed_diff']:+.2f}%")
print(f"Permutation test p-value: {perm_result['p_value']:.4f}")
print(f"Cohen's d effect size: {effect_size:.3f}")
print()
if perm_result["p_value"] < 0.05:
    print("→ Statistically significant difference (p < 0.05)")
else:
    print("→ No statistically significant difference (p ≥ 0.05)")

Resolution Rate Comparison (Mean ± 95% CI)
haiku-4-5             45.45% (43.45% - 47.64%)
haiku-4-5-altool      49.27% (47.09% - 51.64%)
------------------------------------------------------------
Difference (altool - baseline): +3.82%
Permutation test p-value: 0.0539
Cohen's d effect size: 1.003

→ No statistically significant difference (p ≥ 0.05)


In [5]:
# Visualize: CI comparison and permutation distribution
fig = make_subplots(rows=1, cols=2, subplot_titles=("Resolution Rate with 95% CI", "Permutation Test Distribution"))

# Left: CI comparison
setups = [BASELINE_NAME, ALTOOL_NAME]
means = [baseline_boot["mean"], altool_boot["mean"]]
ci_lows = [baseline_boot["ci_low"], altool_boot["ci_low"]]
ci_highs = [baseline_boot["ci_high"], altool_boot["ci_high"]]
colors = ["steelblue", "coral"]

fig.add_trace(
    go.Bar(
        x=setups,
        y=means,
        error_y={"type": "data", "symmetric": False, "array": [h - m for h, m in zip(ci_highs, means, strict=False)], "arrayminus": [m - low for m, low in zip(means, ci_lows, strict=False)]},
        marker_color=colors,
        text=[f"{m:.1f}%" for m in means],
        textposition="outside",
    ),
    row=1,
    col=1,
)

# Right: Permutation distribution
fig.add_trace(
    go.Histogram(x=perm_result["perm_diffs"], nbinsx=50, name="Permutation diffs", marker_color="gray"),
    row=1,
    col=2,
)
fig.add_vline(x=perm_result["observed_diff"], line_dash="dash", line_color="red", row=1, col=2, annotation_text=f"Observed: {perm_result['observed_diff']:.2f}%")
fig.add_vline(x=0, line_dash="dot", line_color="black", row=1, col=2)

fig.update_yaxes(title_text="Resolution Rate (%)", row=1, col=1)
fig.update_xaxes(title_text="Difference in Means (%)", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_layout(height=400, title_text=f"Resolution Rate: {BASELINE_NAME} vs {ALTOOL_NAME}", showlegend=False)
fig.show()

## 2. Pass Metrics Comparison

Compare pass@k (measures the likelihood that an agent gets at least one correct solution in k attempts), and pass^k (measures the probability that all k trials succeed).

In [6]:
baseline_metrics = compute_pass_metrics(baseline_df, k=K_RUNS)
altool_metrics = compute_pass_metrics(altool_df, k=K_RUNS)

print(f"Pass Metrics Comparison (k={K_RUNS} runs)")
print("=" * 65)
print(f"{'Metric':<25} {BASELINE_NAME:>20} {ALTOOL_NAME:>20}")
print("-" * 65)
print(f"{'Mean %':<25} {baseline_metrics['mean_pct']:>19.2f}% {altool_metrics['mean_pct']:>19.2f}%")
print(f"{'pass@k (1-C(n-c,k)/C(n,k))':<25} {baseline_metrics['pass_at_k'] * 100:>19.2f}% {altool_metrics['pass_at_k'] * 100:>19.2f}%")
print(f"{'pass^k (C(c,k)/C(n,k))':<25} {baseline_metrics['pass_hat_k'] * 100:>19.2f}% {altool_metrics['pass_hat_k'] * 100:>19.2f}%")
print("-" * 65)
print(f"{'Instances':<25} {baseline_metrics['n_instances']:>20} {altool_metrics['n_instances']:>20}")

Pass Metrics Comparison (k=10 runs)
Metric                               haiku-4-5     haiku-4-5-altool
-----------------------------------------------------------------
Mean %                                  45.45%               49.27%
pass@k (1-C(n-c,k)/C(n,k))               74.55%               85.45%
pass^k (C(c,k)/C(n,k))                  21.82%               18.18%
-----------------------------------------------------------------
Instances                                   55                   55


In [7]:
# Visualize pass metrics comparison
metrics_names = ["Mean", f"pass@{K_RUNS}", f"pass^{K_RUNS}"]
baseline_vals = [baseline_metrics["mean_pct"], baseline_metrics["pass_at_k"] * 100, baseline_metrics["pass_hat_k"] * 100]
altool_vals = [altool_metrics["mean_pct"], altool_metrics["pass_at_k"] * 100, altool_metrics["pass_hat_k"] * 100]

fig = go.Figure()
fig.add_trace(go.Bar(name=BASELINE_NAME, x=metrics_names, y=baseline_vals, marker_color="steelblue", text=[f"{v:.1f}%" for v in baseline_vals], textposition="outside"))
fig.add_trace(go.Bar(name=ALTOOL_NAME, x=metrics_names, y=altool_vals, marker_color="coral", text=[f"{v:.1f}%" for v in altool_vals], textposition="outside"))

fig.update_layout(
    barmode="group",
    title_text=f"Pass Metrics: {BASELINE_NAME} vs {ALTOOL_NAME}",
    yaxis_title="Percentage (%)",
    height=450,
    yaxis_range=[0, max(*baseline_vals, *altool_vals) + 10],
)
fig.show()

## 3. Tool Usage Analysis

Compare tool usage patterns between the two setups. The altool setup has access to the **AL MCP Server** which provides tools with the `altool-` prefix.

In [8]:
# Expand tool_usage dict into DataFrames for analysis
baseline_tool_df = expand_tool_usage(baseline_df_k)
altool_tool_df = expand_tool_usage(altool_df_k)

baseline_tools = list(baseline_tool_df.columns)
altool_tools = list(altool_tool_df.columns)

# Dynamically detect AL MCP Server tools by prefix
AL_MCP_PREFIX = "altool-"
al_mcp_tools = sorted([t for t in altool_tools if t.startswith(AL_MCP_PREFIX)])

print(f"Tools in {BASELINE_NAME}: {sorted(baseline_tools)}")
print(f"Tools in {ALTOOL_NAME}: {sorted(altool_tools)}")
print(f"\n🔧 AL MCP Server tools (prefix '{AL_MCP_PREFIX}'): {al_mcp_tools}")
print(f"   Other new tools: {sorted(set(altool_tools) - set(baseline_tools) - set(al_mcp_tools))}")

Tools in haiku-4-5: ['bash', 'create', 'edit', 'glob', 'grep', 'powershell', 'read_powershell', 'report_intent', 'stop_powershell', 'view']
Tools in haiku-4-5-altool: ['altool-al_symbolsearch', 'altool-compile', 'bash', 'bash -n', 'create', 'edit', 'glob', 'grep', 'list_powershell', 'powershell', 'read_powershell', 'report_intent', 'stop_powershell', 'view']

🔧 AL MCP Server tools (prefix 'altool-'): ['altool-al_symbolsearch', 'altool-compile']
   Other new tools: ['bash -n', 'list_powershell']


In [9]:
# Compare mean tool usage
all_tools = sorted(set(baseline_tools) | set(altool_tools))

tool_comparison = []
for tool in all_tools:
    baseline_mean = baseline_tool_df[tool].mean() if tool in baseline_tool_df.columns else 0
    altool_mean = altool_tool_df[tool].mean() if tool in altool_tool_df.columns else 0
    is_al_mcp = tool.startswith(AL_MCP_PREFIX)
    tool_comparison.append(
        {
            "Tool": tool,
            "AL MCP": "✓" if is_al_mcp else "",
            f"{BASELINE_NAME}": round(baseline_mean, 2),
            f"{ALTOOL_NAME}": round(altool_mean, 2),
        }
    )

tool_df = pd.DataFrame(tool_comparison).sort_values(ALTOOL_NAME, ascending=False).reset_index(drop=True)
print("Mean Tool Calls per Instance")
print("=" * 75)
tool_df

Mean Tool Calls per Instance


,Tool,AL MCP,haiku-4-5,haiku-4-5-altool
0,view,,18.11,26.86
1,grep,,11.35,18.68
2,powershell,,4.24,7.36
3,glob,,4.66,6.09
4,edit,,4.07,5.63
5,report_intent,,1.80,2.78
6,altool-compile,✓,0.00,0.98
7,stop_powershell,,0.08,0.13
8,altool-al_symbolsearch,✓,0.00,0.06
9,read_powershell,,0.02,0.05


In [10]:
# Visualize tool usage differences
fig = go.Figure()

fig.add_trace(
    go.Bar(
        name=BASELINE_NAME,
        y=tool_df["Tool"],
        x=tool_df[BASELINE_NAME],
        orientation="h",
        marker_color="steelblue",
    )
)
fig.add_trace(
    go.Bar(
        name=ALTOOL_NAME,
        y=tool_df["Tool"],
        x=tool_df[ALTOOL_NAME],
        orientation="h",
        marker_color="coral",
    )
)

fig.update_layout(
    barmode="group",
    title_text="Mean Tool Calls per Instance",
    xaxis_title="Mean Calls",
    height=max(400, len(all_tools) * 35),
    yaxis={"categoryorder": "array", "categoryarray": tool_df["Tool"].tolist()[::-1]},
)
fig.show()

In [11]:
# Focus on AL MCP Server tools usage and correlation with success
if al_mcp_tools:
    print("AL MCP Server Tool Usage Analysis")
    print("=" * 60)

    for tool in al_mcp_tools:
        if tool in altool_tool_df.columns:
            tool_counts = altool_tool_df[tool]
            print(f"\n📊 {tool}")
            print(f"   Total calls: {tool_counts.sum():.0f}")
            print(f"   Mean per instance: {tool_counts.mean():.2f}")
            print(f"   Used in {(tool_counts > 0).sum()}/{len(tool_counts)} records ({(tool_counts > 0).mean() * 100:.1f}%)")

    def plot_tool_usage_analysis(tool_name: str, tool_df: pd.DataFrame, result_df: pd.DataFrame):
        """Plot usage distribution and resolution rate for a tool."""
        if tool_name not in tool_df.columns:
            print(f"Tool {tool_name} not found")
            return

        analysis_df = pd.DataFrame(
            {
                "tool_count": tool_df[tool_name],
                "resolved": result_df["resolved"].values,
            }
        )
        usage_vs_success = (
            analysis_df.groupby("tool_count")
            .agg(
                count=("resolved", "count"),
                resolved_count=("resolved", "sum"),
                resolution_rate=("resolved", "mean"),
            )
            .reset_index()
        )

        total_instances = len(analysis_df)
        usage_vs_success["usage_pct"] = usage_vs_success["count"] / total_instances * 100
        usage_vs_success["resolution_rate"] *= 100

        fig = make_subplots(rows=1, cols=2, subplot_titles=(f"{tool_name}: Usage Distribution", f"{tool_name}: Resolution Rate by Usage"))

        fig.add_trace(
            go.Bar(x=usage_vs_success["tool_count"], y=usage_vs_success["usage_pct"], marker_color="coral"),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Bar(x=usage_vs_success["tool_count"], y=usage_vs_success["resolution_rate"], marker_color="green"),
            row=1,
            col=2,
        )

        fig.update_xaxes(title_text="Number of calls", row=1, col=1)
        fig.update_yaxes(title_text="% of Instances", row=1, col=1)
        fig.update_xaxes(title_text="Number of calls", row=1, col=2)
        fig.update_yaxes(title_text="Resolution Rate (%)", row=1, col=2)
        fig.update_layout(height=400, title_text=f"AL MCP Server: {tool_name} Analysis", showlegend=False)
        fig.show()

    for tool in al_mcp_tools:
        plot_tool_usage_analysis(tool, altool_tool_df, altool_df_k)
else:
    print("No AL MCP Server tools found")

AL MCP Server Tool Usage Analysis

📊 altool-al_symbolsearch
   Total calls: 33
   Mean per instance: 0.06
   Used in 25/550 records (4.5%)

📊 altool-compile
   Total calls: 540
   Mean per instance: 0.98
   Used in 219/550 records (39.8%)


## 4. Instance-Level Agreement Analysis

Which instances are solved by one setup but not the other? This helps identify where the altool MCP server helps or hurts.

In [12]:
# Build instance-level resolution rates for each setup
def compute_instance_resolution_rates(df: pd.DataFrame) -> pd.Series:
    """Compute resolution rate per instance across all runs."""
    return df.groupby("instance_id")["resolved"].mean()


baseline_instance_rates = compute_instance_resolution_rates(baseline_df_k)
altool_instance_rates = compute_instance_resolution_rates(altool_df_k)

# Merge for comparison
instance_comparison = pd.DataFrame(
    {
        "baseline_rate": baseline_instance_rates,
        "altool_rate": altool_instance_rates,
    }
).fillna(0)

instance_comparison["diff"] = instance_comparison["altool_rate"] - instance_comparison["baseline_rate"]
instance_comparison["baseline_any"] = instance_comparison["baseline_rate"] > 0
instance_comparison["altool_any"] = instance_comparison["altool_rate"] > 0

In [13]:
# Agreement matrix
both_solved = ((instance_comparison["baseline_any"]) & (instance_comparison["altool_any"])).sum()
only_baseline = ((instance_comparison["baseline_any"]) & (~instance_comparison["altool_any"])).sum()
only_altool = ((~instance_comparison["baseline_any"]) & (instance_comparison["altool_any"])).sum()
neither = ((~instance_comparison["baseline_any"]) & (~instance_comparison["altool_any"])).sum()

total = len(instance_comparison)

# Visualize agreement
fig = go.Figure(
    data=go.Heatmap(
        z=[[both_solved, only_baseline], [only_altool, neither]],
        x=["Altool Solved", "Altool Failed"],
        y=["Baseline Solved", "Baseline Failed"],
        text=[
            [f"{both_solved}\n({both_solved / total * 100:.1f}%)", f"{only_baseline}\n({only_baseline / total * 100:.1f}%)"],
            [f"{only_altool}\n({only_altool / total * 100:.1f}%)", f"{neither}\n({neither / total * 100:.1f}%)"],
        ],
        texttemplate="%{text}",
        textfont={"size": 14},
        colorscale="Blues",
        showscale=False,
    )
)
fig.update_layout(
    title_text="Instance Agreement Matrix",
    height=400,
    width=500,
)
fig.show()

In [14]:
# Show instances with biggest improvement/regression
instance_comparison_sorted = instance_comparison.sort_values("diff", ascending=False)

# Get project info from the original dataframe
instance_to_project = altool_df_k.drop_duplicates("instance_id").set_index("instance_id")["project"]


def show_top_instances(df: pd.DataFrame, title: str, n: int = 3):
    print(title)
    print("=" * 90)
    display_df = df.head(n).copy()
    display_df["project"] = display_df.index.map(instance_to_project)
    display_df["baseline_rate"] = (display_df["baseline_rate"] * 100).round(1)
    display_df["altool_rate"] = (display_df["altool_rate"] * 100).round(1)
    display_df["diff"] = (display_df["diff"] * 100).round(1)
    display_df = display_df.reset_index()
    print(
        display_df[["instance_id", "project", "baseline_rate", "altool_rate", "diff"]]
        .rename(columns={"instance_id": "Instance", "project": "Project", "baseline_rate": "Baseline %", "altool_rate": "Altool %", "diff": "Δ %"})
        .to_string(index=False)
    )
    print()


show_top_instances(instance_comparison_sorted, "Top 3 Improved Instances (altool better than baseline)", n=3)
show_top_instances(instance_comparison_sorted.iloc[::-1], "Top 3 Regressed Instances (baseline better than altool)", n=3)

Top 3 Improved Instances (altool better than baseline)
                     Instance Project  Baseline %  Altool %  Δ %
microsoftInternal__NAV-193649 Shopify        30.0      90.0 60.0
microsoftInternal__NAV-226448 BaseApp        10.0      40.0 30.0
microsoftInternal__NAV-219082 BaseApp        70.0     100.0 30.0

Top 3 Regressed Instances (baseline better than altool)
                     Instance Project  Baseline %  Altool %   Δ %
microsoftInternal__NAV-218786 BaseApp        50.0      10.0 -40.0
microsoftInternal__NAV-224447 BaseApp        60.0      20.0 -40.0
microsoftInternal__NAV-218995 BaseApp       100.0      70.0 -30.0



## 5. Token Usage & LLM Metrics

Compare token consumption and LLM-related metrics between setups.

In [15]:
# Token and LLM metrics comparison
# These are the metrics from AgentMetrics (excluding tool_usage which is now a dict)
llm_metrics = ["prompt_tokens", "completion_tokens", "turn_count", "llm_duration", "execution_time"]
available_llm_metrics = [m for m in llm_metrics if m in baseline_df_k.columns]

print("Token Usage & LLM Metrics Comparison (Mean ± 95% Bootstrap CI)")
print("=" * 85)
print(f"{'Metric':<20} {BASELINE_NAME:>28} {ALTOOL_NAME:>28}")
print("-" * 85)

llm_results = []
for metric in available_llm_metrics:
    baseline_vals = baseline_df_k[metric].dropna().values
    altool_vals = altool_df_k[metric].dropna().values

    if len(baseline_vals) > 0 and len(altool_vals) > 0:
        baseline_boot = bootstrap_ci(np.array(baseline_vals))
        altool_boot = bootstrap_ci(np.array(altool_vals))

        baseline_str = f"{baseline_boot['mean']:.0f} ({baseline_boot['ci_low']:.0f}-{baseline_boot['ci_high']:.0f})"
        altool_str = f"{altool_boot['mean']:.0f} ({altool_boot['ci_low']:.0f}-{altool_boot['ci_high']:.0f})"
        diff_pct = (altool_boot["mean"] - baseline_boot["mean"]) / baseline_boot["mean"] * 100 if baseline_boot["mean"] > 0 else 0
        print(f"{metric:<20} {baseline_str:>28} {altool_str:>28} ({diff_pct:+.1f}%)")

        llm_results.append(
            {
                "metric": metric,
                "baseline_mean": baseline_boot["mean"],
                "altool_mean": altool_boot["mean"],
                "diff_pct": diff_pct,
            }
        )

Token Usage & LLM Metrics Comparison (Mean ± 95% Bootstrap CI)
Metric                                  haiku-4-5             haiku-4-5-altool
-------------------------------------------------------------------------------------
prompt_tokens           2374117 (2211669-2544315)    3543369 (3287541-3799437) (+49.2%)
completion_tokens              10426 (9825-11035)          17239 (15909-18633) (+65.3%)
turn_count                             41 (39-43)                   64 (60-69) (+57.9%)
llm_duration                        118 (111-125)                191 (177-206) (+62.5%)
execution_time                      151 (143-160)                269 (248-290) (+77.4%)


In [16]:
# Visualize token usage comparison
if llm_results:
    fig = make_subplots(rows=1, cols=len(llm_results), subplot_titles=[r["metric"] for r in llm_results])

    for i, result in enumerate(llm_results, 1):
        metric = result["metric"]
        baseline_vals = baseline_df_k[metric].dropna().values
        altool_vals = altool_df_k[metric].dropna().values

        fig.add_trace(go.Box(y=baseline_vals, name=BASELINE_NAME, marker_color="steelblue", showlegend=(i == 1)), row=1, col=i)
        fig.add_trace(go.Box(y=altool_vals, name=ALTOOL_NAME, marker_color="coral", showlegend=(i == 1)), row=1, col=i)

    fig.update_layout(height=400, title_text="Token Usage & LLM Metrics Distribution")
    fig.show()